# Dependencies

In [ ]:
# Dependencies 
from gazpacho import Soup
from bs4 import BeautifulSoup
import requests
import pandas as pd
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
from splinter import Browser
from webdriver_manager.chrome import ChromeDriverManager

# Web Scrape csv information

In [ ]:
# Run Chrome to assist with identifying another set of html calls.
executable_path = {'executable_path': ChromeDriverManager().install()}
browser = Browser('chrome', **executable_path, headless=False)

In [ ]:
# Visit the new URL for preview
url = "https://s3.amazonaws.com/tripdata/index.html"
browser.visit(url)

In [ ]:
# Requests
html_page = requests.get(url)
soup = BeautifulSoup(html_page.content, "html.parser")

In [ ]:
# HTML object
html = browser.html
# Parse HTML with Beautiful Soup
soup = BeautifulSoup(html, 'html.parser')

# Create an list of csv links
rows = []

# Loop through the links to append 
for link in soup.find_all('a'):
    
    # Append the zip "clean" links
    rows.append(link.get('href'))

# Only capture the zip files that are the focus.
links = rows[92:98]

# Print to confirm the changes
links

In [ ]:
# Close out browser session
browser.quit()

# Unzip and load csv data into a single DataFrame.

## Note:  This takes some computing power to execute.

In [ ]:
# Start a DataFrame
master_df = pd.DataFrame()

# Only do one at a time due to memory limitation
for link in links:
    # open url
    resp = urlopen(link)
    # read zip
    zipfile = ZipFile(BytesIO(resp.read()))
    # Get csv file name
    fname = zipfile.namelist()[0]
    # Data Frame
    df = pd.read_csv(zipfile.open(fname), dtype=object)
    # Close zip
    zipfile.close()
    # Concatenate DataFrame
    master_df = pd.concat([master_df, df], ignore_index=True, sort=False)
    # Print Confirmation
    print(f'{fname} has been completed')
    
# Save Data Frame for Tableau Visuals
master_df.to_csv("bike_rental.csv")

# Print Save Confirmation
print("document has been saved")

In [ ]:
# Free up Memory by deleting a DataFrame no longer needed
del df

# Data Cleansing

## Assess the data

In [ ]:
# Preview Data
master_df.head(5)

In [ ]:
# Length of DataFrame for considering how many rows would be lost for specific columns.
master_df.shape[0]

In [ ]:
# Count Null Values by column
master_df.isnull().sum(axis=0)

## Cleaning the Data such as dropping duplicate columns, removing null values, and changing the datatypes.

In [ ]:
# Drop Duplicate Columns that are primarily empty anyway.
master_df = master_df.drop(master_df.columns[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]], axis=1)

In [ ]:
# Confirm that columns were dropped.
master_df.isnull().sum(axis=0)

In [ ]:
# After seeing updated preview, dropping null values in other columns
master_df = master_df.dropna(subset=["ride_id", "rideable_type", "member_casual", "start_lat", "start_lng", 
                                     "end_lat", "end_lng", "started_at", "ended_at", "start_station_name", 
                                     "start_station_id", "end_station_name", "end_station_id"])

In [ ]:
# Confirm that nulls were dropped.
master_df.isnull().sum(axis=0)

In [ ]:
# Now that we have a clean data set, the data types need to be reviewed for accuracy.
master_df.dtypes

In [ ]:
# New Dependency
import datetime

# Convert datatype for started_at to datetime
master_df["started_at"].astype('datetime64[s]')

In [ ]:
# Convert datatype for ended_at to datetime
master_df["ended_at"].astype('datetime64[s]')

# Export the Clean csv to use for Tableau Visualizations.

In [ ]:
# Save Data Frame for Tableau Visuals
master_df.to_csv("bike_rental.csv")